# NrtkRobustness Analytics Store Demo

This notebook demonstrates the end-to-end workflow for the NrtkRobustness capability with the analytics store:

1. Load a dataset, model, and metric
2. Run the robustness capability (perturbation sweep)
3. Extract per-point scalar records
4. Write to the analytics store
5. Query the robustness curve via SQL

## Setup

In [ ]:
import tempfile
from pathlib import Path

from checkmaite.core.analytics_store import AnalyticsStore, ParquetBackend
from checkmaite.core._common.nrtk_robustness_capability import NrtkRobustnessRecord

## Load Dataset

NRTK requires a dataset, a model, and a metric. Running the full capability
requires model inference, so this demo constructs records directly to
demonstrate the analytics store workflow.

In [ ]:
data_root = Path("../../..")
coco_dir = data_root / "tests" / "data_for_tests" / "coco_resized_val2017"

from checkmaite.core.object_detection.dataset_loaders import CocoDetectionDataset

dataset = CocoDetectionDataset(
    root=str(coco_dir),
    ann_file=str(coco_dir / "instances_val2017_resized_6.json"),
)

print(f"Dataset ID: {dataset.metadata['id']}")
print(f"Images: {len(dataset)}")

## Simulated Robustness Curve

In practice, `extract()` produces these records automatically from the capability
run. Here we construct them directly to demonstrate the multi-record pattern:
one record per (theta_value, metric_key) pair.

This simulates a brightness perturbation sweep with 5 theta values and 2 metrics.

In [ ]:
dataset_id = dataset.metadata["id"]

# Simulate a brightness sweep: 5 theta points x 2 metrics = 10 records
curve_data = [
    (1.0, 0.85, 0.82), (2.0, 0.78, 0.74), (3.0, 0.65, 0.61),
    (4.0, 0.52, 0.48), (5.0, 0.41, 0.37),
]

records = []
for i, (theta, accuracy, f1) in enumerate(curve_data):
    records.append(NrtkRobustnessRecord(
        run_uid="demo_nrtk_run",
        dataset_id=dataset_id,
        model_id="resnet50",
        metric_id="coco_metrics",
        perturber_class="BrightnessPerturber",
        perturber_type="Brightness Perturber",
        theta_key="factor",
        theta_index=i,
        theta_value=theta,
        metric_key="accuracy",
        metric_value=accuracy,
        is_primary=True,
    ))
    records.append(NrtkRobustnessRecord(
        run_uid="demo_nrtk_run",
        dataset_id=dataset_id,
        model_id="resnet50",
        metric_id="coco_metrics",
        perturber_class="BrightnessPerturber",
        perturber_type="Brightness Perturber",
        theta_key="factor",
        theta_index=i,
        theta_value=theta,
        metric_key="f1_score",
        metric_value=f1,
        is_primary=False,
    ))

print(f"Created {len(records)} records ({len(curve_data)} theta points x 2 metrics)")
for r in records[:4]:
    print(f"  theta={r.theta_value}, {r.metric_key}={r.metric_value:.2f}, primary={r.is_primary}")
print(f"  ...")

## Write to Analytics Store

In [ ]:
store_dir = tempfile.mkdtemp(prefix="nrtk_demo_")
backend = ParquetBackend(store_dir)

backend.write(records)

print(f"Store path: {store_dir}")
print(f"Tables: {backend.list_tables()}")
print(f"\nSchema: {backend.describe_table('nrtk_robustness')}")

## Query via SQL

The per-point records enable full robustness curve reconstruction and cross-run comparison.

In [ ]:
# Reconstruct the primary robustness curve
df = backend.query_sql("""
    SELECT theta_value, metric_key, metric_value
    FROM nrtk_robustness
    WHERE is_primary = true
    ORDER BY theta_index
""")
print("Primary robustness curve:")
print(df)

In [ ]:
# Find perturbation levels where performance drops below a threshold
df = backend.query_sql("""
    SELECT theta_value, metric_value
    FROM nrtk_robustness
    WHERE is_primary = true AND metric_value < 0.6
    ORDER BY theta_index
""")
print("Perturbation levels below threshold 0.6:")
print(df)

In [ ]:
# Summary statistics across the curve
df = backend.query_sql("""
    SELECT
        perturber_type,
        model_id,
        metric_key,
        MIN(metric_value) AS min_score,
        MAX(metric_value) AS max_score,
        AVG(metric_value) AS avg_score
    FROM nrtk_robustness
    GROUP BY perturber_type, model_id, metric_key
""")
print("Curve summary by metric:")
print(df)

## Cross-Capability JOIN Example

The `dataset_id` field enables direct JOINs with other capability tables:

```sql
SELECT
    n.model_id,
    n.theta_value,
    n.metric_value,
    f.ber_upper
FROM nrtk_robustness n
JOIN dataeval_feasibility f ON n.dataset_id = f.dataset_id
WHERE n.is_primary = true
```